# BPNet vs Cherimoya: profile JSD and log-counts Pearson comparison

Compares two genome-wide benchmark metrics between BPNet and an older version of Cherimoya models across all ENCODE PRO-cap atlas experiments:

- **Profile JSD** — Jensen-Shannon distance between predicted and observed read profiles across peak loci. Lower is better (0 = perfect, 1 = maximally wrong).
- **Log-counts Pearson** — Pearson correlation between log-predicted and log-observed total read counts per peak. Higher is better.

Metrics are taken from the `genome_wide` field of each per-experiment benchmark JSON, which aggregates predictions across all 7 held-out test chromosome folds.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
import matplotlib.pyplot as plt

## Load data

Both TSVs are generated by consolidating per-experiment benchmark JSONs:
- `performance_metrics/bpnet/procap-atlas_performance_metrics.tsv`
- `performance_metrics/cherimoya/procap-atlas_performance_metrics.tsv` (generated by `src/cherimoya/benchmark/consolidate_metrics.py`)

Only experiments present in both TSVs are included.

In [ ]:
bpnet = pd.read_csv(REPO_ROOT / "performance_metrics" / "bpnet" / "procap-atlas_performance_metrics.tsv", sep="\t")
cherimoya = pd.read_csv(REPO_ROOT / "performance_metrics" / "cherimoya" / "v0.0.1" / "procap-atlas_performance_metrics.tsv", sep="\t")

# Inner join: keeps only experiments benchmarked for both models
df = bpnet[["experiment", "biosample", "total_reads", "profile_jsd", "log_counts_pearson"]].merge(
    cherimoya[["experiment", "profile_jsd", "log_counts_pearson"]],
    on="experiment",
    suffixes=("_bpnet", "_cherimoya"),
)
# Positive delta_jsd = Cherimoya is worse; negative = Cherimoya is better
df["delta_jsd"] = df["profile_jsd_cherimoya"] - df["profile_jsd_bpnet"]
# Positive delta_pearson = Cherimoya is better; negative = Cherimoya is worse
df["delta_pearson"] = df["log_counts_pearson_cherimoya"] - df["log_counts_pearson_bpnet"]

print(f"{len(df)} experiments")
df.head()

## Profile JSD

Each point is one experiment. Points **below** the diagonal indicate experiments where Cherimoya achieves lower (better) JSD than BPNet. Points are colored by read depth to show whether improvements are concentrated in well-covered experiments.

The Wilcoxon signed-rank test assesses whether the JSD values are systematically different between the two models across all experiments.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# --- Scatter ---
ax = axes[0]
sc = ax.scatter(
    df["profile_jsd_bpnet"],
    df["profile_jsd_cherimoya"],
    c=np.log10(df["total_reads"]),
    cmap="viridis",
    s=20,
    alpha=0.8,
    linewidths=0,
)
plt.colorbar(sc, ax=ax, label="log10(total reads)")

# Diagonal reference line (y = x): points below = Cherimoya better
lims = [
    min(df["profile_jsd_bpnet"].min(), df["profile_jsd_cherimoya"].min()) - 0.01,
    max(df["profile_jsd_bpnet"].max(), df["profile_jsd_cherimoya"].max()) + 0.01,
]
ax.plot(lims, lims, color="gray", linestyle="dashed", linewidth=1, zorder=0)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_aspect("equal")
ax.set_xlabel("BPNet profile JSD")
ax.set_ylabel("Cherimoya profile JSD")
ax.set_title("Profile JSD")

stat, pval = wilcoxon(df["profile_jsd_bpnet"], df["profile_jsd_cherimoya"])
n_better = (df["delta_jsd"] < 0).sum()
ax.text(
    0.05, 0.95,
    f"Cherimoya better: {n_better}/{len(df)}\nWilcoxon p={pval:.2e}",
    transform=ax.transAxes, va="top", fontsize=9,
)

# --- Histogram of deltas ---
ax = axes[1]
ax.axvline(0, color="gray", linestyle="dashed", linewidth=1)
ax.hist(df["delta_jsd"], bins=30, color="steelblue", edgecolor="white", linewidth=0.5)
ax.set_xlabel("ΔJSD (Cherimoya − BPNet)")
ax.set_ylabel("Experiments")
ax.set_title("Change in profile JSD")

fig.tight_layout()
fig.savefig(PLOTS_DIR / "bpnet_vs_cherimoya_old_jsd.pdf", dpi=150)
plt.show()

print(f"Median ΔJSD: {df['delta_jsd'].median():.4f}")
print(f"Mean ΔJSD:   {df['delta_jsd'].mean():.4f}")

## Log-counts Pearson

Each point is one experiment. Points **above** the diagonal indicate experiments where Cherimoya achieves higher (better) log-counts Pearson than BPNet. Points are colored by read depth.

The Wilcoxon signed-rank test assesses whether the correlations are systematically different between the two models.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# --- Scatter ---
ax = axes[0]
sc = ax.scatter(
    df["log_counts_pearson_bpnet"],
    df["log_counts_pearson_cherimoya"],
    c=np.log10(df["total_reads"]),
    cmap="viridis",
    s=20,
    alpha=0.8,
    linewidths=0,
)
plt.colorbar(sc, ax=ax, label="log10(total reads)")

# Diagonal reference line (y = x): points above = Cherimoya better
lims = [
    min(df["log_counts_pearson_bpnet"].min(), df["log_counts_pearson_cherimoya"].min()) - 0.01,
    max(df["log_counts_pearson_bpnet"].max(), df["log_counts_pearson_cherimoya"].max()) + 0.01,
]
ax.plot(lims, lims, color="gray", linestyle="dashed", linewidth=1, zorder=0)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_aspect("equal")
ax.set_xlabel("BPNet log-counts Pearson")
ax.set_ylabel("Cherimoya log-counts Pearson")
ax.set_title("Log-counts Pearson")

stat, pval = wilcoxon(df["log_counts_pearson_bpnet"], df["log_counts_pearson_cherimoya"])
n_better = (df["delta_pearson"] > 0).sum()
ax.text(
    0.05, 0.95,
    f"Cherimoya better: {n_better}/{len(df)}\nWilcoxon p={pval:.2e}",
    transform=ax.transAxes, va="top", fontsize=9,
)

# --- Histogram of deltas ---
ax = axes[1]
ax.axvline(0, color="gray", linestyle="dashed", linewidth=1)
ax.hist(df["delta_pearson"], bins=30, color="steelblue", edgecolor="white", linewidth=0.5)
ax.set_xlabel("ΔPearson (Cherimoya − BPNet)")
ax.set_ylabel("Experiments")
ax.set_title("Change in log-counts Pearson")

fig.tight_layout()
fig.savefig(PLOTS_DIR / "bpnet_vs_cherimoya_old_log_counts_pearson.pdf", dpi=150)
plt.show()

print(f"Median ΔPearson: {df['delta_pearson'].median():.4f}")
print(f"Mean ΔPearson:   {df['delta_pearson'].mean():.4f}")